In [2]:
# Import necessary libraries
import os
os.chdir("..")
import json
import torch
from transformers import AutoTokenizer, EsmForProteinFolding
from config import cfg

In [3]:
device = cfg["misc"]["device"]
model_name = cfg["folding_model"]["name"]
model = EsmForProteinFolding.from_pretrained(model_name).to(device)
tokeniser = AutoTokenizer.from_pretrained(model_name)
model.eval()
print(f"Device: {device}")

Loading weights: 100%|██████████| 4533/4533 [00:08<00:00, 538.23it/s, Materializing param=trunk.trunk2sm_z.weight]                                      
EsmForProteinFolding LOAD REPORT from: facebook/esmfold_v1
Key                                | Status     | 
-----------------------------------+------------+-
esm.embeddings.position_ids        | UNEXPECTED | 
esm.contact_head.regression.weight | MISSING    | 
esm.contact_head.regression.bias   | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Device: cuda


In [4]:
# Load a sample fragmented protein
with open(cfg["data"]["fragmented_ecoli"]) as f:
    sample = json.loads(f.readline())

fragments = sample["fragments"]
original = sample["ecoli_original"]
print(f"Original Protein: {original}")
print(f"Original Length: {len(original)}")
print(f"Fragmented Protein: {fragments}")
print(f"Number of Fragments: {len(fragments)}")

Original Protein: MERCGWVSQDPLYIAYHDNEWGVPETDSKKLFEMICLEGQQAGLSWITVLKKRENYRACFHQFDPVKVAAMQEEDVERLVQDAGIIRHRGKIQAIIGNARAYLQMEQNGEPFVDFVWSFVNHQPQVTQATTLSEIPTSTSASDALSKALKKRGFKFVGTTICYSFMQACGLVNDHVVGCCCYPGNKP
Original Length: 187
Fragmented Protein: ['ALK', 'AYLQMEQNGEPFVDFVWSFVNHQPQVTQATTLSEIPTSTSASDALSK', 'K', 'LVQDAGIIR', 'R', 'K', 'GFK', 'ENYR', 'IQAIIGNAR', 'R', 'HR', 'CGWVSQDPLYIAYHDNEWGVPETDSK', 'GK', 'K', 'FVGTTICYSFMQACGLVNDHVVGCCCYPGNKP', 'ACFHQFDPVK', 'VAAMQEEDVER', 'MER', 'LFEMICLEGQQAGLSWITVLK']
Number of Fragments: 19


In [ ]:
def score_protein_confidence(protein):
    inputs = tokeniser([protein], return_tensors="pt", add_special_tokens=False)
    inputs = {k: v.to(device) for k, v in inputs.items()}
    with torch.no_grad():
        outputs = model(**inputs)
    # pLDDT is confidence on a 0-100 scale per residue
    return float(outputs.plddt[0].mean().item())

fragment_scores = [score_protein_confidence(fragment) for fragment in fragments]
weighted_sum = sum(score * len(fragment) for score, fragment in zip(fragment_scores, fragments))
weighted_mean_score = weighted_sum / sum(len(fragment) for fragment in fragments)

print(f"Fragment confidence scores: {[round(s, 2) for s in fragment_scores]}")
print(f"Length-weighted mean fragment pLDDT={weighted_mean_score:.2f}")

Protein length=187 mean_plddt=0.82
